In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("CEREBRAS_API_KEY")


## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-oss-120b",
    base_url="https://api.cerebras.ai/v1",
    api_key=os.environ["CEREBRAS_API_KEY"],
)

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user seeks whimsical factual information about “Lunapolis,” the fictional capital of the moon, including its capital status, weather, population of cheese miners, and potential union strike.\n\n## SUMMARY\n- Identified “Lunapolis” as the capital of the moon.  \n- Reported fictional weather: clear skies, high of 120\u202f°C, low of –100\u202f°C.  \n- Stated that 100,000 cheese miners reside in Lunapolis.  \n- Predicted a likely strike by the cheese miners' union due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone – the user’s inquiries have been answered.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='0f2c2609-b3f8-43a8-a31b-176681db2bf4'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user seeks whimsical factual information about “Lunapolis,” the fictional capital of the moon, including its capital status, weather, population of cheese miners, and potential union strike.

## SUMMARY
- Identified “Lunapolis” as the capital of the moon.  
- Reported fictional weather: clear skies, high of 120 °C, low of –100 °C.  
- Stated that 100,000 cheese miners reside in Lunapolis.  
- Predicted a likely strike by the cheese miners' union due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None – the user’s inquiries have been answered.


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='9d16585e-7b1c-400e-b38f-a7b55e2f81c4'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='e13544d3-5c11-41d2-a0c4-61e4c0186ce8', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='95a632af-1f14-4f23-93e5-107428d8f298'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='6b0875c3-efbc-4ef0-aeaa-63d05df31fab', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='ae225a28-5c3d-4b5c-bd80-df89472bcb8a'),
              AIMessage(content='I’m not able to read the temperature of your hardware remotely

In [8]:
print(response["messages"][-1].content)

I’m not able to read the temperature of your hardware remotely, but you can check it yourself (and, if it’s running hot, take some immediate safety steps). Here’s a quick guide you can follow:

---

## 1. **Quick “feel‑test”**
1. **Power‑off** the device (press the power button for a few seconds if there’s a hard‑reset, or unplug it).  
2. **Touch the chassis** (especially near the power supply, vents, or any metal heat‑sink).  
   - **Warm to the touch** (≈30‑40 °C / 85‑105 °F) is normal for many devices that have been on for a while.  
   - **Hot or scorching** (≥50 °C / 120 °F) suggests it’s overheating.

If it feels hot, **unplug it right away** and let it cool for at least 10‑15 minutes before proceeding.

---

## 2. **Use a thermometer or IR sensor (best method)**
| Tool | How to use | Typical safe range |
|------|------------|--------------------|
| **Digital handheld IR thermometer** | Point the sensor at the surface (about 1‑2 inches away) and press the trigger. | 30‑45 °C (86